# Assignment 13: reversible LLM training, full scale (Colab GPU)

This notebook runs the assignment at the specified scale on one Colab GPU (written for a T4, 16 GB, fp16):

| stage | what it does |
|---|---|
| data | tokenizes ~56M FineWeb-Edu tokens (GPT-2 BPE) |
| check | reversible backward == ordinary autograd, on CPU (float64) and GPU (fp16) |
| screen | 5M-token runs of baseline + 4 reversible variants; picks the variant |
| probe | one-step max batch, memory vs depth, throughput vs batch |
| run 1 | baseline, **20.1M params, 50M tokens**, batch 64 |
| run 2 | chosen reversible variant, same everything, batch 64 |
| run 3 | chosen reversible variant at its **sustained maximum batch**, 50M tokens |
| run 4 | baseline at **its** sustained maximum batch, 50M tokens |
| report | tables + plots in `results/summary.md` and `results/plots/` |

**Sustained maximum batch.** The probe finds the largest batch that survives one training step. Runs 3 and 4 start a full 50M-token run at exactly that batch; if it runs out of memory at any point, it restarts 5% smaller, until a whole run completes. Every attempt is logged in `results/logs/sustained_batch.csv`.

**Surviving disconnects.** Results and the tokenized data are written to Google Drive (`MyDrive/chintana_assignment13/`) as they are produced. If the runtime disconnects, reconnect and run all cells again: finished stages are skipped.

**Time.** Roughly 2–4 hours on a T4 in total (estimate; the 50M-token runs dominate). Each stage is its own cell, so progress is visible.

All logic lives in the scripts under `experiments/assignment13/` (`run_all.py` drives them); this notebook only calls it.

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO_URL = "https://github.com/keshavamurali/chintana.git"
BRANCH = "assignment13"
REPO = "/content/chintana"
USE_DRIVE = True   # False: keep results on the Colab disk only (lost on disconnect)

if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1

# Colab already has a CUDA build of torch. Don't `pip install -e .`: pyproject.toml pins
# torch==2.2.2 for an Intel Mac and would replace it.
!pip -q install tiktoken omegaconf datasets

A13 = "experiments/assignment13"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/chintana_assignment13"
else:
    STORE = "/content/a13_store"
os.makedirs(f"{STORE}/results", exist_ok=True)
os.makedirs(f"{STORE}/data/fineweb_edu", exist_ok=True)
os.makedirs(f"{A13}/data", exist_ok=True)
if not os.path.islink(f"{A13}/data/fineweb_edu"):
    os.symlink(f"{STORE}/data/fineweb_edu", f"{A13}/data/fineweb_edu")

os.environ["PYTHONPATH"] = REPO
RESULTS = f"{STORE}/results"
os.environ["A13_RESULTS_DIR"] = RESULTS   # every script writes its results here
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("results ->", RESULTS)

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

In [ ]:
def stage(name, extra=""):
    # one run_all.py stage; finished runs are skipped, so re-running a cell is safe
    !python {A13}/run_all.py --stages {name} {extra}

## 1. Data: FineWeb-Edu, GPT-2 BPE (~5-10 min)

In [ ]:
stage("data")

## 2. Correctness checks (~1 min)

In [ ]:
stage("check")

## 3. Variant screening: 5 trunks x 5M tokens, batch 64

In [ ]:
stage("screen")

In [ ]:
import json, glob
rows = [json.load(open(p)) for p in sorted(glob.glob(f"{RESULTS}/runs/screen_*/summary.json"))]
for r in rows:
    print(f"{r['variant']:12s} val {r['final_val_loss']:.4f}  train {r['final_train_loss']:.4f}  "
          f"{r['tokens_per_s']:>9,.0f} tok/s  peak {r.get('peak_mem_mb', 0)/1024:.2f} GB  diverged={r['diverged']}")

## 4. Memory and throughput probes

This step finds the largest batch that survives one full training step, for the baseline and the chosen variant. Each is measured with the chunked loss head used in every run (`fused`) and with the standard full-logits head (`standard`). It also measures tokens/s at powers of two up to each maximum, and peak memory vs depth.

In [ ]:
stage("probe")

In [ ]:
!cat {RESULTS}/logs/max_batch.csv

## 5. Run 1: baseline, batch 64, 50M tokens

In [ ]:
stage("run1")

## 6. Run 2: chosen reversible variant, batch 64, 50M tokens

In [ ]:
stage("run2")

## 7. Run 3: chosen reversible variant at its sustained maximum batch, 50M tokens

In [ ]:
stage("run3")

## 8. Run 4: baseline at its sustained maximum batch, 50M tokens

In [ ]:
stage("run4")

In [ ]:
!cat {RESULTS}/logs/sustained_batch.csv

## 9. Report

In [ ]:
stage("report")

In [ ]:
from IPython.display import Markdown, Image, display
R = RESULTS
display(Markdown(open(f"{R}/summary.md").read()))
for name in ["val_loss", "train_loss", "throughput", "peak_memory", "screen_val_loss",
             "memory_vs_depth", "memory_vs_batch", "throughput_vs_batch"]:
    path = f"{R}/plots/{name}.png"
    if os.path.exists(path):
        display(Image(path))

## 10. Download the results

The results folder (CSVs, JSON logs, plots, `summary.md`) is small. It is already in Drive; this also downloads a zip. Put its contents in `experiments/assignment13/results/` in the repo.

In [ ]:
!cd {RESULTS}/.. && zip -qr /content/assignment13_results.zip results
from google.colab import files
files.download("/content/assignment13_results.zip")